# 01 — Data Exploration

This notebook explores the GenImage dataset used for AI-generated image detection.

Contents:
- Image counts per class per generator
- Sample images (real vs fake) from each generator
- Sample DCT maps (real vs fake)
- Class balance verification

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from src.config import Config
from src.transforms import compute_dct_map
from src.seed import fix_seeds

fix_seeds(42)
sns.set_style('whitegrid')
config = Config()

print(f'Data directory: {config.data_dir}')

## 1. Dataset Statistics

In [ ]:
# Load manifest
manifest_path = config.project_root / 'data' / 'data_manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

print('=== Dataset Summary ===')
print(f'Source: {manifest["source"]}')
print(f'Generators: {manifest["generators"]}')
print(f'Image size: {manifest["image_size"]}')
print(f'Total disk usage: {manifest["total_disk_usage_mb"]:.0f} MB')
print()

for split_name, split_data in manifest['splits'].items():
    if split_name == 'test':
        total = sum(sum(v.values()) for v in split_data.values() if v)
        print(f'{split_name}: {total} images across {len([v for v in split_data.values() if v])} generators')
        for gen, counts in split_data.items():
            if counts:
                print(f'  {gen}: {counts}')
    else:
        total = sum(split_data.values())
        print(f'{split_name}: {total} images — {split_data}')

## 2. Sample Images: Real vs Fake (by Generator)

In [ ]:
generators = manifest['generators']

fig, axes = plt.subplots(len(generators), 4, figsize=(16, 4 * len(generators)))
fig.suptitle('Real vs Fake Images by Generator', fontsize=16, y=1.01)

for i, gen in enumerate(generators):
    test_dir = config.data_dir / 'test' / gen
    
    # Get sample real and fake images
    real_imgs = sorted((test_dir / 'real').glob('*.jpg'))[:2]
    fake_imgs = sorted((test_dir / 'fake').glob('*.jpg'))[:2]
    
    for j, img_path in enumerate(real_imgs):
        img = Image.open(img_path)
        axes[i, j].imshow(img)
        axes[i, j].set_title(f'{gen} — Real', fontsize=10)
        axes[i, j].axis('off')
    
    for j, img_path in enumerate(fake_imgs):
        img = Image.open(img_path)
        axes[i, j + 2].imshow(img)
        axes[i, j + 2].set_title(f'{gen} — Fake (AI)', fontsize=10)
        axes[i, j + 2].axis('off')

plt.tight_layout()
plt.savefig(str(config.results_dir / 'plots' / 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/plots/sample_images.png')

## 3. DCT Frequency Maps: Real vs Fake

In [ ]:
fig, axes = plt.subplots(len(generators), 4, figsize=(16, 4 * len(generators)))
fig.suptitle('DCT Frequency Maps: Real vs Fake', fontsize=16, y=1.01)

for i, gen in enumerate(generators):
    test_dir = config.data_dir / 'test' / gen
    
    real_imgs = sorted((test_dir / 'real').glob('*.jpg'))[:2]
    fake_imgs = sorted((test_dir / 'fake').glob('*.jpg'))[:2]
    
    for j, img_path in enumerate(real_imgs):
        img = Image.open(img_path)
        dct = compute_dct_map(img)
        axes[i, j].imshow(dct, cmap='viridis')
        axes[i, j].set_title(f'{gen} — Real DCT', fontsize=10)
        axes[i, j].axis('off')
    
    for j, img_path in enumerate(fake_imgs):
        img = Image.open(img_path)
        dct = compute_dct_map(img)
        axes[i, j + 2].imshow(dct, cmap='viridis')
        axes[i, j + 2].set_title(f'{gen} — Fake DCT', fontsize=10)
        axes[i, j + 2].axis('off')

plt.tight_layout()
plt.savefig(str(config.results_dir / 'plots' / 'dct_maps.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/plots/dct_maps.png')

## 4. Generator Distribution in Training Data

In [ ]:
# Count generators in training data
train_dir = config.data_dir / 'train'
gen_counts = Counter()

for label in ['real', 'fake']:
    for img_path in (train_dir / label).glob('*.jpg'):
        gen = img_path.stem.rsplit('_', 1)[0]
        gen_counts[gen] += 1

gens = sorted(gen_counts.keys())
counts = [gen_counts[g] for g in gens]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(gens, counts, color=sns.color_palette('Set2', len(gens)))
ax.set_xlabel('Generator')
ax.set_ylabel('Image Count')
ax.set_title('Training Data Distribution by Generator')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 200,
            f'{count:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(str(config.results_dir / 'plots' / 'generator_distribution.png'), dpi=150)
plt.show()

## 5. Class Balance Verification

In [ ]:
print('=== Class Balance ===')
for split in ['train', 'val']:
    split_dir = config.data_dir / split
    real_count = len(list((split_dir / 'real').glob('*.jpg')))
    fake_count = len(list((split_dir / 'fake').glob('*.jpg')))
    total = real_count + fake_count
    print(f'{split}: real={real_count} ({real_count/total*100:.1f}%), fake={fake_count} ({fake_count/total*100:.1f}%)')

print('\nTest split (per generator):')
for gen in generators:
    gen_dir = config.data_dir / 'test' / gen
    if gen_dir.exists():
        real_count = len(list((gen_dir / 'real').glob('*.jpg')))
        fake_count = len(list((gen_dir / 'fake').glob('*.jpg')))
        print(f'  {gen}: real={real_count}, fake={fake_count}')

## 6. DataLoader Verification

In [ ]:
from src.dataset import AIDetectDCTDataset
from torch.utils.data import DataLoader

ds = AIDetectDCTDataset(data_dir=config.data_dir, split='train')
loader = DataLoader(ds, batch_size=4, shuffle=True, num_workers=0)
batch = next(iter(loader))

print(f'Dataset size: {len(ds)}')
print(f'RGB shape: {batch["image"].shape}')     
print(f'DCT shape: {batch["dct_map"].shape}')    
print(f'Labels: {batch["label"]}')
print(f'Generators: {batch["metadata"]["generator"]}')
print(f'RGB range: [{batch["image"].min():.2f}, {batch["image"].max():.2f}]')
print(f'DCT range: [{batch["dct_map"].min():.2f}, {batch["dct_map"].max():.2f}]')
print('\n✅ DataLoader working correctly!')